In [ ]:
# ══════════════════════════════════════════════════════════════
# 전체 라벨링 코드 (RetinaFace 없이 OpenCV DNN + MediaPipe)
# ══════════════════════════════════════════════════════════════
import os, cv2, json, shutil, numpy as np, urllib.request
from tqdm import tqdm
from datetime import datetime
from hsemotion_onnx.facial_emotions import HSEmotionRecognizer

# ================= 설정 =================
INPUT_DIR        = "/content/drive/MyDrive/project_3_images/img(sadness)/sadness"
OUTPUT_DIR       = "/content/drive/MyDrive/project_3_images/labeled"
CHECKPOINT_EVERY = 50
MODEL_NAME       = "enet_b0_8_best_afew"
# ========================================

EMOTIONS = ["Anger","Contempt","Disgust","Fear","Happiness","Neutral","Sadness","Surprise"]
EMOTION_CATEGORIES = [{"id": i, "name": e, "supercategory": "emotion"} for i, e in enumerate(EMOTIONS)]
EMOTION_TO_ID      = {e: i for i, e in enumerate(EMOTIONS)}

IMAGES_DIR      = os.path.join(OUTPUT_DIR, "images")
MASKS_DIR       = os.path.join(OUTPUT_DIR, "masks")
LANDMARKS_DIR   = os.path.join(OUTPUT_DIR, "landmarks")
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "label_checkpoint.json")

for d in [IMAGES_DIR, MASKS_DIR, LANDMARKS_DIR]:
    os.makedirs(d, exist_ok=True)


# ── 체크포인트 ────────────────────────────────────────────────
def save_checkpoint(processed_files, coco_data):
    tmp = CHECKPOINT_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump({"processed_files": processed_files, "coco_data": coco_data,
                   "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}, f, ensure_ascii=False)
    os.replace(tmp, CHECKPOINT_PATH)

def load_checkpoint():
    if not os.path.exists(CHECKPOINT_PATH):
        return None
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        ckpt = json.load(f)
    print(f"[체크포인트 발견] {ckpt['saved_at']} — {len(ckpt['processed_files'])}장 완료")
    return ckpt


# ── OpenCV DNN 얼굴 감지기 로드 ───────────────────────────────
print("얼굴 감지 모델 다운로드 중...")
urllib.request.urlretrieve(
    "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
    "/content/face_model.caffemodel"
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
    "/content/deploy.prototxt"
)
net = cv2.dnn.readNetFromCaffe("/content/deploy.prototxt", "/content/face_model.caffemodel")
print("✅ 얼굴 감지 모델 로드 완료")

def detect_faces_dnn(img_bgr, conf_threshold=0.5):
    h, w = img_bgr.shape[:2]
    blob = cv2.dnn.blobFromImage(cv2.resize(img_bgr, (300, 300)), 1.0,
                                  (300, 300), (104.0, 177.0, 123.0))
    net.setInput(blob)
    detections = net.forward()
    faces = []
    for i in range(detections.shape[2]):
        conf = detections[0, 0, i, 2]
        if conf > conf_threshold:
            box = detections[0, 0, i, 3:7] * [w, h, w, h]
            x1, y1, x2, y2 = box.astype(int)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            if x2 > x1 and y2 > y1:
                faces.append((x1, y1, x2 - x1, y2 - y1))
    return faces


# ── MediaPipe 랜드마크 추출 ───────────────────────────────────
import mediapipe as mp
mp_face_mesh = mp.solutions.face_mesh

LEFT_EYE   = [33,7,163,144,145,153,154,155,133,173,157,158,159,160,161,246]
RIGHT_EYE  = [362,382,381,380,374,373,390,249,263,466,388,387,386,385,384,398]
NOSE_IDX   = [1,2,4,5,6,19,20,48,64,98,115,218,219,220,237,238,239,241,242,
              278,294,327,344,438,439,440,457,458,459,461,462]
MOUTH_IDX  = [61,84,17,314,405,321,375,291,185,40,39,37,0,267,269,270,409,291]
FACE_OVAL  = [10,338,297,332,284,251,389,356,454,323,361,288,397,365,379,378,
              400,377,152,148,176,149,150,136,172,58,132,93,234,127,162,21,54,103,67,109]

def extract_landmarks(img_bgr, x, y, w, h):
    # ✅ 핵심 수정: 얼굴 crop 대신 전체 이미지 입력 (MediaPipe가 더 잘 감지)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    ih, iw  = img_bgr.shape[:2]

    with mp_face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.3
    ) as face_mesh:
        results = face_mesh.process(img_rgb)

    if not results.multi_face_landmarks:
        # 전체 이미지도 실패 시 → crop 이미지로 재시도
        face_rgb = cv2.cvtColor(img_bgr[y:y+h, x:x+w], cv2.COLOR_BGR2RGB)
        # 너무 작으면 리사이즈 후 시도
        fh, fw = face_rgb.shape[:2]
        if fw < 100 or fh < 100:
            scale    = 200 / min(fw, fh)
            face_rgb = cv2.resize(face_rgb, (int(fw*scale), int(fh*scale)))

        with mp_face_mesh.FaceMesh(
            static_image_mode=True,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.2
        ) as face_mesh:
            results = face_mesh.process(face_rgb)

        if not results.multi_face_landmarks:
            return None

        # crop 기준 좌표 → 원본 이미지 기준으로 변환
        lm   = results.multi_face_landmarks[0].landmark
        rh, rw = face_rgb.shape[:2]
        def to_abs_crop(idx):
            return (int(lm[idx].x * rw * (fw / rw)) + x,
                    int(lm[idx].y * rh * (fh / rh)) + y)
        def all_pts_crop():
            return [(int(l.x * rw * (fw/rw)) + x,
                     int(l.y * rh * (fh/rh)) + y) for l in lm]

        return {
            "all":     all_pts_crop(),
            "eyes":    [to_abs_crop(i) for i in LEFT_EYE + RIGHT_EYE],
            "nose":    [to_abs_crop(i) for i in NOSE_IDX],
            "mouth":   [to_abs_crop(i) for i in MOUTH_IDX],
            "contour": [to_abs_crop(i) for i in FACE_OVAL]
        }

    # 전체 이미지 기준 좌표
    lm = results.multi_face_landmarks[0].landmark
    def to_abs(idx):
        return (int(lm[idx].x * iw), int(lm[idx].y * ih))

    return {
        "all":     [(int(l.x*iw), int(l.y*ih)) for l in lm],
        "eyes":    [to_abs(i) for i in LEFT_EYE + RIGHT_EYE],
        "nose":    [to_abs(i) for i in NOSE_IDX],
        "mouth":   [to_abs(i) for i in MOUTH_IDX],
        "contour": [to_abs(i) for i in FACE_OVAL]
    }


# ── 마스크 & 시각화 ───────────────────────────────────────────
def create_face_mask(img_shape, contour_pts):
    mask = np.zeros(img_shape[:2], dtype=np.uint8)
    if contour_pts and len(contour_pts) > 2:
        cv2.fillPoly(mask, [np.array(contour_pts, dtype=np.int32)], 255)
    return mask

def draw_landmarks(img, landmarks):
    vis = img.copy()
    for part, color in [("eyes",(0,255,255)), ("nose",(255,100,0)),
                         ("mouth",(0,100,255)), ("contour",(0,255,0))]:
        for (px, py) in landmarks.get(part, []):
            cv2.circle(vis, (px, py), 2, color, -1)
    return vis


# ── 메인 ─────────────────────────────────────────────────────
def run_labeling():
    print(f"감정 모델 로드 중: {MODEL_NAME}")
    fer   = HSEmotionRecognizer(model_name=MODEL_NAME)
    print("✅ 감정 모델 로드 완료\n")

    valid_ext   = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    image_files = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_ext)])
    total       = len(image_files)

    ckpt = load_checkpoint()
    if ckpt:
        processed_set = set(ckpt["processed_files"])
        coco_data     = ckpt["coco_data"]
        image_files   = [f for f in image_files if f not in processed_set]
        print(f"남은 작업: {len(image_files)}장 / 전체 {total}장\n")
    else:
        processed_set = set()
        coco_data = {
            "info":        {"description": "Face Emotion Dataset", "version": "1.0",
                            "date_created": datetime.now().strftime("%Y-%m-%d")},
            "licenses":    [],
            "categories":  EMOTION_CATEGORIES,
            "images":      [],
            "annotations": []
        }
        print(f"총 {total}장 라벨링 시작\n")

    image_id      = len(coco_data["images"])
    annotation_id = len(coco_data["annotations"])
    batch_counter = 0
    stats = {"success": 0, "no_face": 0, "no_landmark": 0, "error": 0}

    for filename in tqdm(image_files, desc="라벨링", unit="장"):
        img_path = os.path.join(INPUT_DIR, filename)
        try:
            img_bgr = cv2.imread(img_path)
            if img_bgr is None:
                raise ValueError("이미지 읽기 실패")

            img_h, img_w = img_bgr.shape[:2]
            shutil.copy2(img_path, os.path.join(IMAGES_DIR, filename))

            # 얼굴 감지 (conf 낮춰서 최대한 잡기)
            faces = detect_faces_dnn(img_bgr, conf_threshold=0.4)
            if not faces:
                # Haar Cascade로 fallback
                cascade = cv2.CascadeClassifier(
                    cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
                f = cascade.detectMultiScale(
                    cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY),
                    scaleFactor=1.1, minNeighbors=3, minSize=(30, 30))
                faces = list(f) if len(f) > 0 else []

            if not faces:
                stats["no_face"] += 1
                processed_set.add(filename)
                batch_counter += 1
                continue

            x, y, w, h = max(faces, key=lambda f: f[2] * f[3])

            # 감정 분석
            face_crop     = img_bgr[y:y+h, x:x+w]
            face_rgb      = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
            emotion_label, scores = fer.predict_emotions(face_rgb, logits=False)
            confidence    = float(np.max(scores))
            category_id   = EMOTION_TO_ID.get(emotion_label, 5)

            # 랜드마크 (전체 이미지로 시도 → crop으로 재시도)
            landmarks = extract_landmarks(img_bgr, x, y, w, h)
            mask_filename = os.path.splitext(filename)[0] + "_mask.png"

            if landmarks and landmarks["contour"]:
                mask = create_face_mask(img_bgr.shape, landmarks["contour"])
                segmentation = [[float(v) for pt in landmarks["contour"] for v in pt]]
                # 랜드마크 시각화 저장
                lm_img = draw_landmarks(img_bgr, landmarks)
                cv2.imwrite(os.path.join(LANDMARKS_DIR,
                            os.path.splitext(filename)[0] + "_lm.jpg"), lm_img)
            else:
                # bbox로 대체
                mask = np.zeros((img_h, img_w), dtype=np.uint8)
                cv2.rectangle(mask, (x, y), (x+w, y+h), 255, -1)
                segmentation = [[float(x),float(y), float(x+w),float(y),
                                 float(x+w),float(y+h), float(x),float(y+h)]]
                landmarks    = {"eyes": [], "nose": [], "mouth": [], "contour": []}
                stats["no_landmark"] += 1

            cv2.imwrite(os.path.join(MASKS_DIR, mask_filename), mask)

            coco_data["images"].append({
                "id": image_id, "file_name": filename,
                "width": img_w, "height": img_h,
                "mask_file": mask_filename,
                "emotion": emotion_label, "confidence": round(confidence, 4),
                "all_scores": {e: round(float(s), 4) for e, s in zip(EMOTIONS, scores)}
            })
            coco_data["annotations"].append({
                "id": annotation_id, "image_id": image_id,
                "category_id": category_id,
                "bbox": [float(x), float(y), float(w), float(h)],
                "area": float(w * h),
                "segmentation": segmentation,
                "landmarks": {k: landmarks[k] for k in ["eyes","nose","mouth","contour"]},
                "iscrowd": 0, "confidence": round(confidence, 4)
            })

            image_id += 1; annotation_id += 1
            stats["success"] += 1
            processed_set.add(filename)

        except Exception as e:
            stats["error"] += 1
            processed_set.add(filename)
            tqdm.write(f"  [에러] {filename}: {e}")

        batch_counter += 1
        if batch_counter % CHECKPOINT_EVERY == 0:
            save_checkpoint(list(processed_set), coco_data)
            tqdm.write(f"  💾 체크포인트 저장 ({len(processed_set)}/{total}장)")

    # 최종 저장
    save_checkpoint(list(processed_set), coco_data)
    coco_path = os.path.join(OUTPUT_DIR, "coco_annotations.json")
    with open(coco_path, "w", encoding="utf-8") as f:
        json.dump(coco_data, f, indent=2, ensure_ascii=False)

    print(f"""
╔══════════════════════════════════════════╗
║           라벨링 완료 리포트              ║
╠══════════════════════════════════════════╣
║  전체:          {total:>5}장                ║
║  성공:          {stats['success']:>5}장                ║
║  얼굴 미감지:   {stats['no_face']:>5}장                ║
║  랜드마크 실패: {stats['no_landmark']:>5}장  (bbox 대체)    ║
║  에러:          {stats['error']:>5}장                ║
╚══════════════════════════════════════════╝
저장 위치: {OUTPUT_DIR}""")

run_labeling()